Exploring the data

In [87]:
# Importing necessary libraries
import pandas as pd # reading in as a df
import statsmodels.formula.api as smf # to run the OLS

from sklearn import linear_model
from sklearn.linear_model import Lasso, LassoCV, LogisticRegressionCV  # most common lasso
from sklearn.preprocessing import StandardScaler
import hdmpy # this one has rlasso

# Get rid of all warning
import warnings
warnings.filterwarnings('ignore')

In [56]:
# Read the data in
file = "dataverse_files/UCT_FINAL_CLEAN.dta"
data = pd.read_stata(file)

In [57]:
# Quick look
print(f"The shape of the df is {data.shape}")
print(data.describe())

The shape of the df is (2880, 981)
           surveyid    femaleres      maleres      village        treat  \
count  2.880000e+03  2880.000000  2880.000000  2880.000000  2880.000000   
mean   1.568195e+09     0.500000     0.500000   152.438194     0.349306   
min    1.065403e+08     0.000000     0.000000    15.000000     0.000000   
25%    1.172235e+09     0.000000     0.000000    86.000000     0.000000   
50%    1.482530e+09     0.500000     0.500000   129.000000     0.000000   
75%    2.169777e+09     1.000000     1.000000   222.000000     1.000000   
max    2.989871e+09     1.000000     1.000000   303.000000     1.000000   
std    7.270573e+08     0.500087     0.500087    83.579866     0.476833   

         spillover  purecontrol  control_village  treatXfemalerec  \
count  2880.000000  2880.000000      2880.000000      2880.000000   
mean      0.350694     0.300000         0.300000         0.198611   
min       0.000000     0.000000         0.000000         0.000000   
25%       0.0

In [58]:
print(data[['surveyid', 'femaleres', 'maleres', 'village', 'treat', 
            'baselinedate', 'endlinedate', 
            'asset_total_ppp0','asset_total_ppp_full0','asset_total_ppp_miss0','asset_total_ppp1']
            ].head(11))

       surveyid  femaleres  maleres  village  treat baselinedate endlinedate  \
0   106540326.0          0      1.0       74    0.0   2011-04-29  2012-09-04   
1   106540326.0          1      0.0       74    0.0   2011-04-29  2012-09-04   
2   106769148.0          0      1.0       55    1.0   2011-05-19  2012-10-15   
3   106769148.0          1      0.0       55    1.0   2011-05-19  2012-10-15   
4   106799419.0          0      1.0       71    0.0   2011-06-09  2012-10-03   
5   106799419.0          1      0.0       71    0.0   2011-06-09  2012-10-03   
6   106854772.0          0      1.0       46    0.0   2011-05-09  2012-09-07   
7   106854772.0          1      0.0       46    0.0   2011-05-09  2012-09-07   
8   106979214.0          0      1.0       15    1.0   2011-05-26  2012-09-03   
9   106979214.0          1      0.0       15    1.0   2011-05-26  2012-09-03   
10  107079236.0          0      1.0       15    0.0   2011-05-26         NaT   

    asset_total_ppp0  asset_total_ppp_f

In [59]:
for col in data.columns:
    print(col)

surveyid
femaleres
maleres
village
treat
spillover
purecontrol
control_village
treatXfemalerec
treatXmalerec
treatXfemalerecXmarried
treatXmalerecXmarried
treatXsinglerec
treatXlump
treatXmonthly
treatXlarge
treatXsmall
treatXmalerecXlump
treatXmalerecXsmall
treatXlumpXsmall
treatXmalerecXlumpXsmall
treatXmalerecXlarge
treatXlumpXlarge
treatXmalerecXlumpXlarge
treatXmalerecXmonthly
treatXmonthlyXsmall
treatXmalerecXmonthlyXsmall
treatXmonthlyXlarge
treatXmalerecXmonthlyXlarge
treatXfemalerecXlump
treatXfemalerecXsmall
treatXfemalerecXlumpXsmall
treatXfemalerecXlarge
treatXfemalerecXlumpXlarge
treatXfemalerecXmonthly
treatXfemalerecXmonthlyXsmall
treatXfemalerecXmonthlyXlarge
treatXmaleres
treatXfemaleres
treatXlargeXmaleres
treatXlargeXfemaleres
treatXsmallXmaleres
treatXsmallXfemaleres
treatXlumpXmaleres
treatXlumpXfemaleres
treatXmonthlyXmaleres
treatXmonthlyXfemaleres
treatXmalerecXmaleres
treatXmalerecXfemaleres
treatXfemalerecXmaleres
treatXfemalerecXfemaleres
treatXlumpXearly
tre

In [ ]:
# Adding all plausible covariates:

covariate_cols = [
    # Demographics
    'b_age',
    'b_married',
    'b_children',
    'b_hhsize',
    'b_edu',
    'hh_children0',
    'hh_totalmembers0',
    'femaleres',          # gender of respondent

    # Baseline economic outcomes (main indices)
    'asset_total_ppp0',
    'cons_nondurable_ppp0',
    'ent_total_rev_ppp0',
    'fs_hhfoodindexnew0',

    # Baseline asset sub-components
    'asset_livestock_ppp0',
    'asset_durable_ppp0',
    'asset_savings_ppp0',
    'asset_land_owned_total0',
    'asset_niceroof0',

    # Baseline consumption sub-components
    'cons_allfood_ppp_m0',
    'cons_alcohol_ppp_m0',
    'cons_tobacco_ppp_m0',

    # Enterprise
    'ent_wagelabor0',
    'ent_ownfarm0',
    'ent_business0',

    # Financial
    'fin_remittances_rec_ppp0',

    # Wellbeing indices
    'psy_index_z0',
    'med_hh_healthindex0',
    'ed_index0',
    'ih_overall_index_z0',

    # M-Pesa access
    'given_mpesa',

    # Missing dummies
    'asset_total_ppp_miss0',
    'cons_nondurable_ppp_miss0',
    'ent_total_rev_ppp_miss0',
    'fs_hhfoodindexnew_miss0',
    'psy_index_z_miss0',
    'med_hh_healthindex_miss0',
    'ed_index_miss0',
    'ih_overall_index_z_miss0'
]

1. Checking if we have selection bias and as a result whether double Lasso methods are necessary.

In [ ]:
# Data preparation (shared by all methods)
data_clean = data.copy()
D = data_clean['treat'].values

village_dummies = pd.get_dummies(data_clean['village'], prefix='village', drop_first=True)
X = pd.concat([data_clean[covariate_cols], village_dummies], axis=1).fillna(0)
col_names = covariate_cols + list(village_dummies.columns)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X.values)

# Plain Logistic Regression: D on X
data_clean_with_dummies = pd.concat([data_clean, village_dummies], axis=1).fillna(0)
formula = 'treat ~ ' + ' + '.join(col_names)
logit_model = smf.logit(formula, data=data_clean_with_dummies).fit(method='bfgs',disp=0, maxiter=200)

print("PLAIN LOGISTIC REGRESSION: D on X")
print(f"Pseudo R-squared (McFadden): {logit_model.prsquared:.4f}")
print(f"LR test p-value: {logit_model.llr_pvalue:.4f}")
print(f"Observations: {int(logit_model.nobs)}")

# CV Logistic LASSO: D on X 
results_treat_cvlasso = []

lasso_logit = LogisticRegressionCV(
    Cs=20,
    cv=5,
    penalty='l1',
    solver='liblinear',
    random_state=42
)
lasso_logit.fit(X_scaled, D)

selected_cols_cv = [col_names[i] for i, c in enumerate(lasso_logit.coef_[0]) if c != 0]

results_treat_cvlasso.append({
    'Method':  'CV Logistic LASSO',
    'N selected': len(selected_cols_cv),
    'Selected Variables': selected_cols_cv
})

print("\nCV LOGISTIC LASSO: D on X")
print(pd.DataFrame(results_treat_cvlasso).to_string(index=False))

# RLASSO: D on X
results_treat_rlasso = []

X_array = X.values
rlasso_d = hdmpy.rlasso(X_array, D, post=True)

selected_idx     = rlasso_d.est['index'].values.flatten()
selected_cols_rl = [col_names[i] for i, s in enumerate(selected_idx) if s]

results_treat_rlasso.append({
    'Method': 'rlasso',
    'N selected': len(selected_cols_rl),
    'Selected Variables': selected_cols_rl
})

print("\nRLASSO: D on X")
print(pd.DataFrame(results_treat_rlasso).to_string(index=False))

1. Recovering the Baseline Values in Table I

In [60]:
# Try and reproduce the baseline OLS (only done cols 1 and 2 thus far, no interaction treatments)
# (Need to fix outcomes education index, psych index, and female empowerment index)
"""
Basline Balance: y_vhib = a_v + B_0 + B_1.T_vh + e_vhib

- y_vhib: outcome with suffix 0 for value at baseline
- a_v: 'village'
- T_vh: 'treat'
"""

# Keeping only households that were were within village, not village comparisons
data_within = data[data['purecontrol'] == 0].copy()

# Drop duplicate observations (one male and female for each)
data = data_within[data_within['femaleres'] == 1].copy()

# Empty list
results = []

# Define all our outcome variables at baseline
outcomes_base = [
    'asset_total_ppp0',
    'cons_nondurable_ppp0',
    'ent_total_rev_ppp0',
    'fs_hhfoodindexnew0',
    'med_hh_healthindex0',
    'ed_index0',
    'psy_index_z0',
    
    'ih_overall_index_z0'
]

# Attempting to recreate table 1
for outcome in outcomes_base:
    # Run ols for each outcome
    model = smf.ols(f'{outcome} ~ treat + C(village)', data=data).fit()

    # Getting the control means as per table 1 and column 1 
    control_mean = data[data["spillover"] == 1][outcome].mean()
    control_sd = data[data["spillover"] == 1][outcome].std()

    results.append({
        'Outcome': outcome,
        'Control Mean': f"{control_mean:.2f}",
        'Control SD': f"({control_sd:.2f})",
        'Treat Coef': f"{model.params['treat']:.2f}",
        'SE': f"({model.bse['treat']:.2f})",
        'P-value': f"{model.pvalues['treat']:.3f}"
    })

results_df = pd.DataFrame(results) # makes results list into a df
print(results_df.to_string(index=False)) # prints results_df without index




             Outcome Control Mean Control SD Treat Coef      SE P-value
    asset_total_ppp0       383.36   (374.34)      -1.15 (24.36)   0.962
cons_nondurable_ppp0       181.99   (127.23)      -6.16  (8.23)   0.454
  ent_total_rev_ppp0        84.92   (402.79)     -33.19 (18.87)   0.079
  fs_hhfoodindexnew0        -0.00     (1.00)       0.00  (0.06)   0.997
 med_hh_healthindex0         0.01     (1.02)       0.03  (0.06)   0.683
           ed_index0         0.00     (1.00)      -0.07  (0.06)   0.275
        psy_index_z0        -0.00     (1.00)       0.04  (0.07)   0.557
 ih_overall_index_z0         0.00     (1.00)      -0.05  (0.07)   0.484


2. Recovering the ATE from their OLS ANCOVA in Table II

In [61]:
# Try and reproduce the ATE in table 2 (for cols 1 and 2) - within village households
# (Need to fix outcomes education index, psych index, and female empowerment index)
"""
Basline Balance: y_vhiE = a_v + B_0 + B_1.T_vh + delta_1.y_vhib + delta_2.M_vhiB + e_vhiE

- y_vhiE: outcome with suffix 1 at endline (variable of interest)

- a_v: 'village' (village fixed effects)
- B_0: intercept which represents base village (intercept)
- T_vh: 'treat' (treatment)
- y_vhib: outcome with suffix 0 for value at baseline (a baseline control)
- M_vhiB: used as a dummy if baseline value is missing (a baseline control)
"""

# Keeping only households that were were within village, not village comparisons
data_within = data[data['purecontrol'] == 0].copy()

# Drop duplicate observations (one male and female for each)
data = data_within[data_within['femaleres'] == 1].copy()

# Empty list
results = []

# Define all our outcome variables at endline - this is our variable of interest
outcomes_end = [
    'asset_total_ppp1',
    'cons_nondurable_ppp1',
    'ent_total_rev_ppp1',
    'fs_hhfoodindexnew1',
    'med_hh_healthindex1',
    'ed_index1',
    'psy_index_z1',
    'ih_overall_index_z1'
]

# Define all our outcome variables at baseline
outcomes_base = [
    'asset_total_ppp0',
    'cons_nondurable_ppp0',
    'ent_total_rev_ppp0',
    'fs_hhfoodindexnew0',
    'med_hh_healthindex0',
    'ed_index0',
    'psy_index_z0',
    'ih_overall_index_z0'
]

# Dummy for outcomes that are missing at baseline
outcomes_missing = [
    'asset_total_ppp_miss0',
    'cons_nondurable_ppp_miss0',
    'ent_total_rev_ppp_miss0',
    'fs_hhfoodindexnew_miss0',
    'med_hh_healthindex_miss0',
    'ed_index_miss0',
    'psy_index_z_miss0',
    'ih_overall_index_z_miss0'
]

# Drop the 68 missing endline values 
data = data[data['endlinedate'].notna()].copy()

# Attempting to recreate table 1
for outcome_end, outcome_base, outcome_miss in zip(outcomes_end, outcomes_base, outcomes_missing):
    model = smf.ols(f'{outcome_end} ~ C(village) + treat + {outcome_base} + {outcome_miss}', data=data).fit()

    # Getting the control means as per table 1 and column 1 
    control_mean = data[data["spillover"] == 1][outcome_end].mean()
    control_sd = data[data["spillover"] == 1][outcome_end].std()

    results.append({
        'Outcome': outcome_end,
        'Control Mean': f"{control_mean:.2f}",
        'Control SD': f"({control_sd:.2f})",
        'Treat Coef': f"{model.params['treat']:.2f}",
        'SE': f"({model.bse['treat']:.2f})",
        'P-value': f"{model.pvalues['treat']:.3f}"
    })

    # print(model.summary())

results_df = pd.DataFrame(results) # makes results list into a df
print(results_df.to_string(index=False)) # prints results_df without index




             Outcome Control Mean Control SD Treat Coef      SE P-value
    asset_total_ppp1       494.80   (415.32)     301.51 (27.45)   0.000
cons_nondurable_ppp1       157.61    (82.18)      35.66  (5.82)   0.000
  ent_total_rev_ppp1        48.98    (90.52)      16.15  (5.82)   0.006
  fs_hhfoodindexnew1         0.00     (1.00)       0.26  (0.06)   0.000
 med_hh_healthindex1        -0.00     (1.00)      -0.03  (0.06)   0.579
           ed_index1         0.00     (1.00)       0.08  (0.06)   0.152
        psy_index_z1        -0.00     (1.00)       0.23  (0.07)   0.001
 ih_overall_index_z1         0.00     (1.00)      -0.01  (0.07)   0.866


Running_ML_Methods_to_recover_ATE

In [101]:
import warnings
warnings.filterwarnings('ignore')

results_all = []

for outcome_end, outcome_base, outcome_miss in zip(outcomes_end, outcomes_base, outcomes_missing):

    data_clean = data[data[outcome_end].notna()].copy()
    Y = data_clean[outcome_end].values

    village_dummies = pd.get_dummies(
        data_clean['village'], prefix='village', drop_first=True
    )

    X_with_treat = pd.concat([
        data_clean[['treat'] + covariate_cols],
        village_dummies
    ], axis=1).fillna(0)

    col_names_with = ['treat'] + covariate_cols + list(village_dummies.columns)
    data_clean_dummies = pd.concat([data_clean, village_dummies], axis=1)

    # Helper — count only substantive covariates (no village dummies, no treat)
    def count_substantive(selected):
        return sum(
            1 for c in selected
            if not c.startswith('village_') and c != 'treat'
        )

    # Helper — extract only substantive covariate names
    def get_substantive(selected):
        return [c for c in selected if not c.startswith('village_') and c != 'treat']

    # ── (3) NAIVE CV LASSO ────────────────────────────────────
    scaler = StandardScaler()
    X_scaled_with = scaler.fit_transform(X_with_treat.values)

    lasso_naive = linear_model.LassoCV(cv=5, random_state=42, max_iter=10000)
    lasso_naive.fit(X_scaled_with, Y)

    selected_naive      = [col_names_with[i] for i, c in enumerate(lasso_naive.coef_) if c != 0]
    treat_coef_naive    = lasso_naive.coef_[0] / scaler.scale_[0]
    n_selected_naive    = count_substantive(selected_naive)
    substantive_naive   = get_substantive(selected_naive)

    # ── (4) POST CV LASSO ─────────────────────────────────────
    selected_controls_naive = [c for c in selected_naive if c != 'treat']
    selected_post           = ['treat'] + selected_controls_naive

    formula_post    = f"{outcome_end} ~ " + " + ".join(selected_post)
    model_post      = smf.ols(formula_post, data=data_clean_dummies).fit()
    treat_coef_post = model_post.params['treat']
    se_post         = model_post.bse['treat']
    n_selected_post = count_substantive(selected_post)  # same as naive by construction

    # ── (5) ROBUST RLASSO ─────────────────────────────────────
    rlasso_robust       = hdmpy.rlasso(X_with_treat.values, Y, post=True)
    treat_coef_robust   = rlasso_robust.est['coefficients'].values.flatten()[0]
    selected_idx_robust = rlasso_robust.est['index'].values.flatten()
    selected_robust     = [col_names_with[i] for i, s in enumerate(selected_idx_robust) if s]
    n_selected_robust   = count_substantive(selected_robust)
    substantive_robust  = get_substantive(selected_robust)

    # ── (6) POST RLASSO OLS ───────────────────────────────────
    selected_controls_robust = [c for c in selected_robust if c != 'treat']
    selected_rl              = ['treat'] + selected_controls_robust

    formula_rl    = f"{outcome_end} ~ " + " + ".join(selected_rl)
    model_rl      = smf.ols(formula_rl, data=data_clean_dummies).fit()
    treat_coef_rl = model_rl.params['treat']
    se_rl         = model_rl.bse['treat']
    n_selected_rl = count_substantive(selected_rl)  # same as robust by construction

    # ── Print selected substantive covariates per outcome ─────
    print(f"\n{'='*60}")
    print(f"Outcome: {outcome_end}")
    print(f"\n  CV LASSO selected ({n_selected_naive} substantive vars):")
    print(f"  {substantive_naive}")
    print(f"\n  rlasso selected ({n_selected_robust} substantive vars):")
    print(f"  {substantive_robust}")

    results_all.append({
        'Outcome':          outcome_end,
        'Naive Coef':       f"{treat_coef_naive:.4f}",
        'Naive N':          n_selected_naive,
        'Post CV Coef':     f"{treat_coef_post:.4f}",
        'Post CV SE':       f"({se_post:.4f})",
        'Post CV N':        n_selected_post,
        'Robust Coef':      f"{treat_coef_robust:.4f}",
        'Robust N':         n_selected_robust,
        'Post rlasso Coef': f"{treat_coef_rl:.4f}",
        'Post rlasso SE':   f"({se_rl:.4f})",
        'Post rlasso N':    n_selected_rl,
    })

print(f"\n{'='*60}")
results_df = pd.DataFrame(results_all)
print(results_df.to_string(index=False))


Outcome: asset_total_ppp1

  CV LASSO selected (11 substantive vars):
  ['asset_total_ppp0', 'asset_livestock_ppp0', 'asset_durable_ppp0', 'asset_niceroof0', 'cons_allfood_ppp_m0', 'cons_alcohol_ppp_m0', 'ent_ownfarm0', 'ent_business0', 'ed_index0', 'ih_overall_index_z0', 'ih_overall_index_z_miss0']

  rlasso selected (3 substantive vars):
  ['asset_total_ppp0', 'asset_livestock_ppp0', 'asset_niceroof0']

Outcome: cons_nondurable_ppp1

  CV LASSO selected (13 substantive vars):
  ['b_married', 'b_hhsize', 'b_edu', 'hh_totalmembers0', 'asset_total_ppp0', 'cons_nondurable_ppp0', 'fs_hhfoodindexnew0', 'asset_durable_ppp0', 'asset_savings_ppp0', 'ent_business0', 'psy_index_z0', 'ed_index0', 'given_mpesa']

  rlasso selected (4 substantive vars):
  ['b_hhsize', 'asset_total_ppp0', 'cons_nondurable_ppp0', 'asset_durable_ppp0']

Outcome: ent_total_rev_ppp1

  CV LASSO selected (14 substantive vars):
  ['b_hhsize', 'b_edu', 'hh_totalmembers0', 'asset_total_ppp0', 'ent_total_rev_ppp0', 'fs_hhf